In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
GOLD_DATA_PATH = PROJECT_ROOT / "data" / "gold"

RESULTS_FILE = RAW_DATA_PATH / "results.csv"

In [5]:
df_results = pd.read_csv(RESULTS_FILE)

df_results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [6]:
df_results.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [8]:
df_results.shape

(49477, 9)

In [9]:
df_results.isna().sum()

date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [10]:
df_results.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral'], dtype='str')

In [13]:
df_results["date"] = pd.to_datetime(df_results["date"])

df_results["year"] = df_results["date"].dt.year
df_results["month"] = df_results["date"].dt.month
df_results["decade"] = (df_results["year"] // 10) * 10

df_results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,month,decade
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1872,11,1870
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1873,3,1870
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1874,3,1870
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1875,3,1870
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1876,3,1870


In [14]:
df_results["tournament"].value_counts().head(20)

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

In [15]:
df_results["home_team"].nunique(), df_results["away_team"].nunique()

(327, 321)

In [16]:
df_results.groupby("year").size().tail(20)

year
2007     988
2008    1101
2009     925
2010     863
2011    1119
2012    1025
2013     963
2014     856
2015    1039
2016     920
2017     924
2018     929
2019    1149
2020     347
2021    1115
2022     970
2023    1054
2024    1231
2025    1002
2026     383
dtype: int64

In [17]:
df_world_cup = df_results[df_results["tournament"] == "FIFA World Cup"].copy()

df_world_cup.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,month,decade
1490,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930
1491,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930
1492,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930
1493,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930
1494,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930


In [18]:
df_world_cup.shape

(1036, 12)

In [19]:
df_world_cup["year"].value_counts().sort_index()

year
1930    18
1934    17
1938    18
1950    22
1954    26
1958    35
1962    32
1966    32
1970    32
1974    38
1978    38
1982    52
1986    52
1990    52
1994    52
1998    64
2002    64
2006    64
2010    64
2014    64
2018    64
2022    64
2026    72
Name: count, dtype: int64

In [20]:
df_world_cup["total_goals"] = df_world_cup["home_score"] + df_world_cup["away_score"]

df_world_cup["result"] = np.select(
    [
        df_world_cup["home_score"] > df_world_cup["away_score"],
        df_world_cup["home_score"] < df_world_cup["away_score"]
    ],
    [
        "Home Win",
        "Away Win"
    ],
    default="Draw"
)

df_world_cup.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,month,decade,total_goals,result
1490,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930,3.0,Away Win
1491,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930,5.0,Home Win
1492,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930,3.0,Away Win
1493,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930,4.0,Away Win
1494,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True,1930,7,1930,1.0,Home Win


In [21]:
total_matches = len(df_world_cup)
total_goals = df_world_cup["total_goals"].sum()
avg_goals_per_match = df_world_cup["total_goals"].mean()

total_matches, total_goals, avg_goals_per_match

(1036, np.float64(2809.0), np.float64(2.8316532258064515))

In [22]:
df_world_cup.groupby("year").agg(
    matches=("date", "count"),
    goals=("total_goals", "sum"),
    avg_goals_per_match=("total_goals", "mean")
).reset_index()

,year,matches,goals,avg_goals_per_match
0,1930,18,70.0,3.888889
1,1934,17,70.0,4.117647
2,1938,18,84.0,4.666667
3,1950,22,88.0,4.000000
4,1954,26,140.0,5.384615
5,1958,35,126.0,3.600000
6,1962,32,89.0,2.781250
7,1966,32,89.0,2.781250
8,1970,32,95.0,2.968750
9,1974,38,97.0,2.552632


In [23]:
output_file = PROCESSED_DATA_PATH / "world_cup_matches_processed.csv"

df_world_cup.to_csv(output_file, index=False)

output_file

WindowsPath('c:/Users/nicolas.nocente/OneDrive - Inpasa Agroindustrial SA/Documentos/Projetos/world-cup-analytics-command-center/data/processed/world_cup_matches_processed.csv')